# GPU vs FPGA — Rate-Distortion Comparison

Compare inference quality on **GPU** (W&B) and on **FPGA** (ZCU102) across all four architectures
(**ResSHyp**, **SHyp**, **ResFP**, **FP**) — ReLU activation, no output_padding baseline.

**Data sources:**
- **GPU metrics** — loaded from `SAR_DDC_FPGA_all_runs_WandB.csv` (no API call required).
  Regenerate with `python notebooks/fetch_wandb_runs.py` (~1 min).
- **FPGA metrics** — read from `compiled_models/<name>/results/metrics.json` after `batch_deploy.py`.

**Structure:**
1. Setup
2. Load GPU runs → tidy DataFrame
3. Load FPGA results → tidy DataFrame
4. Merge & coverage check
5. Aggregate statistics across seeds
6. RD-curve plots (GPU vs FPGA)
7. FPGA degradation Δ plot
8. BPP comparison (likelihood vs rANS bitstream)
9. Hamburg tile visualization + close-up
10. All-metrics grid

## 1 · Setup & filters

In [ ]:
import json
import sys
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple, Union

import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import numpy as np
import pandas as pd
import torch

ROOT_DIR = Path("..").resolve()
sys.path.insert(0, str(ROOT_DIR))

from src.utils.constants import EPS
from src.utils.metrics import psnr as _src_psnr
from src.utils.processing_utils import clip as clip_logI

# ── Data sources ──────────────────────────────────────────────────────────────
WANDB_CSV = ROOT_DIR / "notebooks" / "SAR_DDC_FPGA_all_runs_WandB.csv"
COMPILED_MODELS_DIR = ROOT_DIR / "results" / "fpga" / "compiled_models"

# ── Seeds ─────────────────────────────────────────────────────────────────────
# Seeds 0-5 are the main training seeds; seed=42 is excluded (manual tests).
ACCEPTED_SEEDS = [0, 1, 2, 3, 4, 5]

# ── Color palette (Okabe-Ito, colorblind-safe) ────────────────────────────────
C = json.load(open(ROOT_DIR / "notebooks" / "plots_colors.json"))

# Per-architecture colors — used in RD-curve plots (color encodes architecture)
ARCH_COLORS = {
    arch: C["architectures"].get(arch, "#999999") for arch in ["ResSHyp", "SHyp", "ResFP", "FP"]
}

# Per-backend colors — used in delta, BPP, and tile visualization
BACKEND_COLORS = {
    "gpu": C["compare_gpu_fpga"]["gpu"],  # "#0072B2" blue
    "fpga": C["compare_gpu_fpga"]["fpga"],  # "#009E73" green
}

# Per-metric colors — used in delta bar chart
METRIC_COLORS = {
    "psnr": C["metrics"]["psnr"],
    "ssim": C["metrics"]["ssim"],
    "epd": C["metrics"]["epd"],
    "error_bars": C["metrics"]["error_bars"],
}

MARKERS = {"gpu": "o", "fpga": "s"}
LINESTYLES = {"gpu": "-", "fpga": "--"}

# ── Output ────────────────────────────────────────────────────────────────────
PLOTS_DIR = ROOT_DIR / "results" / "plots"
SAVE_FIGURES = True  # global toggle — set False to skip all saves
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

# ── ANSI shortcuts ────────────────────────────────────────────────────────────
r, g, b, y, e = "\033[31m", "\033[32m", "\033[34m", "\033[33m", "\033[0m"

## 2 · Load GPU runs → tidy DataFrame

Load matching runs from the pre-built W&B CSV (`SAR_DDC_FPGA_all_runs_WandB.csv`).
No API call — regenerate anytime with `python notebooks/fetch_wandb_runs.py` (~1 min).

One row per run (one `(lambda, seed)` pair). The function filters to the FPGA-deployed
baseline: **ReLU** activation + **no output_padding**. Metrics come from `test_sub500/`.

**Tidy / long format** — one row per `(lambda, seed, backend)`. This makes aggregation,
filtering, and plotting one-liners instead of nested loops.

In [ ]:
def load_gpu_runs_from_csv(
    csv_path: Path = WANDB_CSV,
    archs: Optional[List[str]] = None,
    accepted_seeds: List[int] = ACCEPTED_SEEDS,
    verbose: bool = True,
) -> pd.DataFrame:
    """Load GPU run metrics from the pre-built W&B CSV. No API call required.

    Run ``notebooks/fetch_wandb_runs.py`` to regenerate the CSV (~1 min).
    Metrics are taken from the ``test_sub500/`` prefix (500-patch evaluation set),
    matching the FPGA evaluation split.

    Filters applied (FPGA-deployable baseline):
    1. Dataset:        TSXSSCDataModule only
    2. Seeds:          ``accepted_seeds`` (default 0–5)
    3. Tags:           removes ``debug``, ``crashed``, ``lr_search``
    4. LR:             ResFP/FP runs keep only lr=5e-4 (retrained baseline)
    5. GDN activation: removes all GDN variants (not FPGA-deployable)
    6. Output padding: removes ``no_output_padding=False`` (not FPGA-deployable)
    7. Architecture:   restricts to ``archs`` (default all four)

    Args:
        csv_path:       Path to the W&B runs CSV.
        archs:          Architectures to include; ``None`` loads all four.
        accepted_seeds: Seeds to include.
        verbose:        Print one line per filter step that removes runs.
    """
    if not csv_path.exists():
        raise FileNotFoundError(
            f"CSV not found: {csv_path}\nRun `python notebooks/fetch_wandb_runs.py` first."
        )
    raw = pd.read_csv(csv_path, index_col="id")

    # ── Parse non-scalar columns ──────────────────────────────────────────────
    raw["tags"] = raw["tags"].apply(lambda v: json.loads(v) if isinstance(v, str) else [])
    raw["no_output_padding"] = raw["no_output_padding"].map(
        {"True": True, "False": False, True: True, False: False}
    )
    raw["lmbda"] = pd.to_numeric(raw["lmbda"], errors="coerce")
    raw["seed"] = pd.to_numeric(raw["seed"], errors="coerce")
    raw["lr"] = pd.to_numeric(raw["lr"], errors="coerce").fillna(0.0)

    filt = raw.copy()
    if verbose:
        print(f"CSV total: {len(filt)} runs")

    # ── Filter: dataset ───────────────────────────────────────────────────────
    ACCEPTED_DATASETS = ["TSXSSCDataModule"]
    mask = filt["data_name"].isin(ACCEPTED_DATASETS)
    if verbose and (~mask).any():
        bad = filt[~mask]["data_name"].value_counts().to_dict()
        print(f"  {b}[dataset]{e}        − {(~mask).sum():>4}  (other: {bad})")
    filt = filt[mask].copy()

    # ── Filter: seeds ─────────────────────────────────────────────────────────
    mask = filt["seed"].isin(accepted_seeds)
    if verbose and (~mask).any():
        bad_seeds = sorted(filt[~mask]["seed"].dropna().unique().tolist())
        print(f"  {b}[seeds]{e}          − {(~mask).sum():>4}  (removed seeds: {bad_seeds})")
    filt = filt[mask].copy()

    # ── Filter: bad tags ──────────────────────────────────────────────────────
    TAGS_TO_REMOVE = ["debug", "crashed", "lr_search"]
    for tag in TAGS_TO_REMOVE:
        mask = filt["tags"].apply(lambda ts: tag in ts)
        if mask.any():
            if verbose:
                print(f"  {b}[tag '{tag}']{e}  − {mask.sum():>4}")
            filt = filt[~mask].copy()

    # ── Filter: ResFP/FP bad LR (old runs at 1e-4; retrained baseline at 5e-4) ─
    mask_bad_lr = filt["architecture"].isin(["ResFP", "FP"]) & (filt["lr"] != 5e-4)
    if mask_bad_lr.any():
        if verbose:
            bad_lrs = sorted(filt[mask_bad_lr]["lr"].unique().tolist())
            print(f"  {b}[ResFP/FP lr≠5e-4]{e} − {mask_bad_lr.sum():>4}  (LRs: {bad_lrs})")
        filt = filt[~mask_bad_lr].copy()

    # ── Filter: GDN activation (not FPGA-deployable) ──────────────────────────
    mask_gdn = filt["model_name"].str.contains("gdn", case=False, na=False)
    if mask_gdn.any():
        if verbose:
            print(f"  {b}[GDN activation]{e} − {mask_gdn.sum():>4}  (not FPGA-deployable)")
        filt = filt[~mask_gdn].copy()

    # ── Filter: output_padding ≠ 0 (not FPGA-deployable) ─────────────────────
    mask_op = filt["no_output_padding"] != True  # noqa: E712  (True/False/NaN)
    if mask_op.any():
        if verbose:
            print(
                f"  {b}[output_padding]{e} − {mask_op.sum():>4}  (no_output_padding=False, not FPGA-deployable)"
            )
        filt = filt[~mask_op].copy()

    # ── Filter: architectures ─────────────────────────────────────────────────
    _archs = archs if archs is not None else ["ResSHyp", "SHyp", "ResFP", "FP"]
    mask = filt["architecture"].isin(_archs)
    if verbose and (~mask).any():
        bad = filt[~mask]["architecture"].value_counts().to_dict()
        print(f"  {b}[arch]{e}           − {(~mask).sum():>4}  (other: {bad})")
    filt = filt[mask].copy()

    if verbose:
        print(f"After filtering:  {g}{len(filt)}{e} runs  (expect 240 = 4 archs × 10 λ × 6 seeds)")

    # ── Map to the shared schema used by downstream code ──────────────────────
    def _metric(col: str) -> np.ndarray:
        full = f"test_sub500/{col}"
        if full in filt.columns:
            return pd.to_numeric(filt[full], errors="coerce").values
        return np.full(len(filt), float("nan"))

    result = pd.DataFrame(
        {
            "lambda": filt["lmbda"].values,
            "seed": filt["seed"].astype(int).values,
            "wandb_id": filt.index.values,
            "wandb_name": filt["run_name"].values,
            "backend": "gpu",
            "arch": filt["architecture"].values,
            "run_dir": filt["run_dir"].fillna("").values
            if "run_dir" in filt.columns
            else [""] * len(filt),
            # BPP
            "bpp_likelihood": _metric("bpp"),
            "bpp_bitstream": _metric("bpp_bitstream"),
            # Quality vs MERLIN
            "psnr_merlin": _metric("psnr_merlin"),
            "mse_merlin": _metric("mse_merlin"),
            "ssim_merlin": _metric("ssim_merlin"),
            "ms_ssim_merlin": _metric("ms_ssim_merlin"),
            # Quality vs ADAM-NOC
            "psnr_adam_noc": _metric("psnr_adam_noc"),
            "mse_adam_noc": _metric("mse_adam_noc"),
            "ssim_adam_noc": _metric("ssim_adam_noc"),
            "ms_ssim_adam_noc": _metric("ms_ssim_adam_noc"),
            # SAR quality metrics
            "enl_recon": _metric("enl_recon"),
            "ratio_mean": _metric("ratio_mean"),
            "ratio_enl": _metric("ratio_enl"),
            "epd_merlin": _metric("epd_merlin"),
            "epd_adam_noc": _metric("epd_adam_noc"),
        }
    )
    result = result.sort_values(["lambda", "seed"]).reset_index(drop=True)
    print(f"Loaded {g}{len(result)}{e} GPU runs  ({result['arch'].value_counts().to_dict()})")
    print(f"  {result['lambda'].nunique()} λ values  ×  {result['seed'].nunique()} seeds")
    return result


# ── Config ────────────────────────────────────────────────────────────────────
ARCHS_TO_LOAD = ["ResSHyp", "SHyp", "ResFP", "FP"]  # restrict to a subset if needed
# ─────────────────────────────────────────────────────────────────────────────
gpu_df = load_gpu_runs_from_csv(archs=ARCHS_TO_LOAD)
gpu_df.head(6)

## 3 · Load FPGA results → tidy DataFrame

Scan `compiled_models/` on disk. Each model directory is linked back to a W&B run via
`manifest.json` — first by `wandb_run_id` (present on runs deployed after the fix to
`deploy.py` today), then by `(seed, lambda)` lookup for the 60 existing runs.

**FPGA metrics available:** `bpp` (real rANS bitstream), `psnr` vs Noisy / ADAM / MERLIN.
No SSIM on FPGA yet — those columns are `NaN`. For a future implementation in pure NumPy,
see the comment at the bottom of this cell.


In [ ]:
def _build_wandb_id_lookup(gpu_df: pd.DataFrame) -> Dict[Tuple, str]:
    """Build a (seed, lambda) → wandb_id lookup from the already-loaded GPU DataFrame.

    Used as a fallback for manifests that predate the wandb_run_id field.
    """
    return {(row.seed, row["lambda"]): row.wandb_id for _, row in gpu_df.iterrows()}


def _infer_arch(name: str) -> str:
    """Derive architecture tag from a model or directory name string."""
    for arch in ["ResSHyp", "SHyp", "ResFP", "FP"]:
        if arch in name:
            return arch
    return "unknown"


def load_fpga_results(
    gpu_df: pd.DataFrame,
    archs: Optional[List[str]] = None,
) -> pd.DataFrame:
    """Scan compiled_models/ and return a tidy DataFrame (one row per model with results).

    Filters: seed must be in ACCEPTED_SEEDS (excludes seed=42 and other manual runs).
    Links each FPGA result to a W&B run ID via manifest.json ``wandb_run_id`` if available,
    otherwise via (seed, lambda) lookup from ``gpu_df``.

    FPGA metrics schema (from results/metrics.json):
        {"Noisy":  {"bpp": ..., "mse": ..., "psnr": ..., "ssim": ..., "enl": ...,
                    "ratio_mean": ..., "ratio_enl": ...},
         "ADAM":   {"bpp": ..., "mse": ..., "psnr": ..., "ssim": ..., "epd": ...},
         "MERLIN": {"bpp": ..., "mse": ..., "psnr": ..., "ssim": ..., "epd": ...},
         "recon":  {"enl": ..., "ratio_mean": ..., "ratio_enl": ...}}

    Args:
        gpu_df: Loaded GPU DataFrame — used for (seed, lambda) → wandb_id lookup.
        archs:  Architectures to include; ``None`` includes all found on disk.
    """
    wandb_lookup = _build_wandb_id_lookup(gpu_df)
    rows = []
    skipped = []

    for model_dir in sorted(COMPILED_MODELS_DIR.iterdir()):
        manifest_path = model_dir / "manifest.json"
        metrics_path = model_dir / "results" / "metrics.json"

        if not manifest_path.exists():
            continue
        manifest = json.loads(manifest_path.read_text())

        seed = manifest.get("seed")
        lmbda = manifest.get("lambda")

        if seed not in ACCEPTED_SEEDS:
            skipped.append(model_dir.name)
            continue

        if not metrics_path.exists():
            print(f"  {y}WARNING{e}: no results/metrics.json for {model_dir.name} — skipping")
            continue

        # Prefer the explicit 'architecture' field if deploy.py wrote it; else infer from name
        model_name = manifest.get("model_name", model_dir.name)
        arch = manifest.get("architecture") or _infer_arch(model_name)

        if archs is not None and arch not in archs:
            continue

        metrics = json.loads(metrics_path.read_text())
        wandb_id = manifest.get("wandb_run_id") or wandb_lookup.get((seed, lmbda), "")

        row: Dict[str, Any] = {
            "lambda": lmbda,
            "seed": seed,
            "wandb_id": wandb_id,
            "wandb_name": model_name,
            "backend": "fpga",
            "arch": arch,
            "model_dir": str(model_dir),
            "run_dir": "",  # not applicable for FPGA
            # BPP from real rANS bitstream
            "bpp_likelihood": float("nan"),
            "bpp_bitstream": metrics.get("MERLIN", {}).get("bpp"),
            # Quality vs MERLIN
            "psnr_merlin": metrics.get("MERLIN", {}).get("psnr"),
            "mse_merlin": metrics.get("MERLIN", {}).get("mse"),
            "ssim_merlin": metrics.get("MERLIN", {}).get("ssim"),
            "ms_ssim_merlin": float("nan"),  # not computed on FPGA
            # Quality vs ADAM-NOC
            "psnr_adam_noc": metrics.get("ADAM", {}).get("psnr"),
            "mse_adam_noc": metrics.get("ADAM", {}).get("mse"),
            "ssim_adam_noc": metrics.get("ADAM", {}).get("ssim"),
            "ms_ssim_adam_noc": float("nan"),  # not computed on FPGA
            # SAR quality metrics
            "enl_recon": metrics.get("recon", {}).get("enl"),
            "ratio_mean": metrics.get("recon", {}).get("ratio_mean"),
            "ratio_enl": metrics.get("recon", {}).get("ratio_enl"),
            "epd_merlin": metrics.get("MERLIN", {}).get("epd"),
            "epd_adam_noc": metrics.get("ADAM", {}).get("epd"),
        }
        rows.append(row)

    df = pd.DataFrame(rows).sort_values(["lambda", "seed"]).reset_index(drop=True)
    if skipped:
        print(f"  Skipped {len(skipped)} models outside ACCEPTED_SEEDS")
    print(f"Loaded {g}{len(df)}{e} FPGA results  ({df['arch'].value_counts().to_dict()})")
    print(f"  {df['lambda'].nunique()} λ values  ×  {df['seed'].nunique()} seeds")
    no_link = df[df["wandb_id"] == ""]
    if len(no_link):
        print(f"  {y}WARNING{e}: {len(no_link)} FPGA results could not be linked to a W&B run ID")
    return df


fpga_df = load_fpga_results(gpu_df, archs=ARCHS_TO_LOAD)
fpga_df.head(6)

## 4 · Merge & coverage check

Concatenate GPU and FPGA rows into one DataFrame and verify we have both backends for
every `(lambda, seed)` pair. Missing combinations are shown explicitly.

**`pivot_table` primer** — reshapes long → wide: `index` defines rows, `columns` defines
the new column headers (here `backend`), `values` is the cell content. Think of it as a
spreadsheet cross-tab. Missing combinations appear as `NaN`.


In [ ]:
# Merge both backends into a single tidy DataFrame
df = pd.concat([gpu_df, fpga_df], ignore_index=True)
df["lambda"] = df["lambda"].astype(float)
df["seed"] = df["seed"].astype(int)
df = df.sort_values(["lambda", "seed", "backend"]).reset_index(drop=True)

print(f"Full DataFrame: {len(df)} rows ({df['backend'].value_counts().to_dict()})")
print(f"λ values: {sorted(df['lambda'].unique())}")
print(f"Seeds:    {sorted(df['seed'].unique())}")

# ── Coverage table ────────────────────────────────────────────────────────────
# pivot_table: rows=lambda, columns=backend, cells=count of runs present (should all be 6)
coverage = df.pivot_table(
    index=["lambda", "arch"], columns="backend", values="psnr_merlin", aggfunc="count"
).rename_axis(None, axis=1)

missing_gpu = coverage[coverage["gpu"] < len(ACCEPTED_SEEDS)] if "gpu" in coverage else []
missing_fpga = coverage[coverage["fpga"] < len(ACCEPTED_SEEDS)] if "fpga" in coverage else []

print(f"\nCoverage (expected {len(ACCEPTED_SEEDS)} runs per λ per backend):")
print(coverage.to_string())
if len(missing_gpu):
    print(f"\n{y}WARNING: incomplete GPU coverage:{e}")
    print(missing_gpu)
if len(missing_fpga):
    print(f"\n{y}WARNING: incomplete FPGA coverage:{e}")
    print(missing_fpga)
else:
    print(f"\n{g}✓ Full coverage: all (λ, seed) pairs present on both backends.{e}")


## 5 · Aggregate statistics across seeds

`groupby(["lambda","backend"]).agg(...)` produces a DataFrame indexed by `(lambda, backend)` with a
two-level column `(metric, stat)`. This is the single source of truth for all plots below.

`query("backend == 'gpu'")` — pandas SQL-style row filter. Returns a view (not a copy), so it's
fast and readable. Equivalent to `df[df["backend"] == "gpu"]` but cleaner for compound conditions.


In [ ]:
METRIC_COLS = [
    "bpp_likelihood",
    "bpp_bitstream",
    "psnr_merlin",
    "mse_merlin",
    "ssim_merlin",
    "ms_ssim_merlin",
    "psnr_adam_noc",
    "mse_adam_noc",
    "ssim_adam_noc",
    "ms_ssim_adam_noc",
    # SAR quality metrics
    "enl_recon",
    "ratio_mean",
    "ratio_enl",
    "epd_merlin",
    "epd_adam_noc",
]

# groupby + agg: for each (lambda, backend) group compute mean/std/min/max of every metric.
# Result is a DataFrame with MultiIndex columns: (metric_col, stat) e.g. ("psnr_merlin","mean").
stats_df = (
    df.groupby(["lambda", "backend", "arch"])[METRIC_COLS]
    .agg(["mean", "std", "min", "max"])
    .sort_index()  # sort by (lambda, backend, arch)
)

# ── BPP column selection ──────────────────────────────────────────────────────
# Use bpp_bitstream if ALL runs (both backends) have it; else fall back to
# bpp_likelihood for GPU and bpp_bitstream for FPGA (they are not directly comparable
# but are the best available proxies per backend).
gpu_has_bitstream = df.query("backend == 'gpu'")["bpp_bitstream"].notna().all()
if gpu_has_bitstream:
    BPP_GPU_COL = "bpp_bitstream"
    print(f"{g}✓ All GPU runs have bpp_bitstream — using it for RD-curves.{e}")
else:
    BPP_GPU_COL = "bpp_likelihood"
    n_missing = df.query("backend == 'gpu'")["bpp_bitstream"].isna().sum()
    print(
        f"{y}⚠  {n_missing} GPU runs missing bpp_bitstream — using bpp_likelihood (soft entropy estimate).{e}"
    )
    print("   Run update_wandb_runs.py to compute real bitstream BPP.")

BPP_FPGA_COL = "bpp_bitstream"  # always real rANS on FPGA
print(f"   GPU  BPP column : {BPP_GPU_COL}")
print(f"   FPGA BPP column : {BPP_FPGA_COL}")

# Quick preview: mean PSNR vs MERLIN at each λ for each backend
preview = stats_df["psnr_merlin"]["mean"].unstack(["arch", "backend"])
n_seeds = df["seed"].nunique()
print(f"\nMean PSNR vs MERLIN across {n_seeds} seeds:")
print(preview.to_string(float_format="{:.2f}".format))


## 6 · RD-curve plots — GPU vs FPGA

One curve per `(architecture, backend)` combination.
- **Color** → architecture  |  **Linestyle** → backend (solid=GPU, dashed=FPGA)
- **Band** → min/max range across seeds  |  **Error bars** → ±1 std on BPP

Set `ARCHS_TO_PLOT` in the call cell to control which architectures appear.

In [ ]:
def plot_rd_curve(
    ax: plt.Axes,
    stats: pd.DataFrame,
    backend: str,
    arch: str,
    bpp_col: str,
    quality_col: str,
    label: Optional[str] = None,
    annotate_lambda: bool = False,
) -> None:
    """Plot one RD curve (one backend + architecture) with error band and BPP error bars."""
    bpp_mean = stats[(bpp_col, "mean")].astype(float)
    bpp_std = stats[(bpp_col, "std")].astype(float).fillna(0)
    q_mean = stats[(quality_col, "mean")].astype(float)
    q_min = stats[(quality_col, "min")].astype(float)
    q_max = stats[(quality_col, "max")].astype(float)

    order = bpp_mean.argsort()
    bpp_s = bpp_mean.iloc[order].values
    q_s = q_mean.iloc[order].values
    q_lo = q_min.iloc[order].values
    q_hi = q_max.iloc[order].values
    bpp_std_s = bpp_std.iloc[order].values
    lambdas = stats.index.get_level_values("lambda").to_numpy()[order]

    color = ARCH_COLORS.get(arch, BACKEND_COLORS.get(backend, "#999999"))
    marker = MARKERS[backend]
    ls = LINESTYLES[backend]
    lbl = label or f"{arch} {backend.upper()}"

    ax.plot(
        bpp_s,
        q_s,
        marker=marker,
        linestyle=ls,
        color=color,
        label=lbl,
        linewidth=1.5,
        markersize=5,
    )
    ax.fill_between(bpp_s, q_lo, q_hi, alpha=0.15, color=color)
    ax.errorbar(
        bpp_s,
        q_s,
        xerr=bpp_std_s,
        fmt="none",
        ecolor=METRIC_COLORS["error_bars"],
        alpha=0.5,
        capsize=2,
    )

    if annotate_lambda:
        for x, yv, lam in zip(bpp_s, q_s, lambdas):
            ax.annotate(
                f"λ={int(lam)}",
                (x, yv),
                textcoords="offset points",
                xytext=(4, 3),
                fontsize=7,
                color=METRIC_COLORS["error_bars"],
                alpha=0.8,
            )


def _parse_quality_col(
    quality_col: Union[str, Tuple[str, str]],
) -> Tuple[str, str]:
    """Return (col_key, y_label) from either a plain string or a (key, label) tuple."""
    if isinstance(quality_col, tuple):
        return quality_col[0], quality_col[1]
    return quality_col, quality_col.replace("_", " ")


def make_rd_figure(
    stats_df: pd.DataFrame,
    archs: Optional[List[str]] = None,
    quality_col: Union[str, Tuple[str, str]] = "psnr_merlin",
    bpp_gpu_col: str = "bpp_likelihood",
    bpp_fpga_col: str = "bpp_bitstream",
    annotate_lambda: bool = False,
    title: Optional[str] = None,
    save_name: Optional[str] = None,
    ax: Optional[plt.Axes] = None,
) -> plt.Figure:
    """RD-curve figure: one curve per (architecture, backend) combination.

    Color encodes architecture (Okabe-Ito palette).
    Linestyle encodes backend (GPU=solid, FPGA=dashed).
    Error band = min/max across seeds.  Error bars = ±1 std on BPP.

    Args:
        archs:       Architectures to plot; ``None`` plots all present in ``stats_df``.
        quality_col: Column key string (e.g. ``"psnr_merlin"``) or ``(key, y_label)`` tuple.
        save_name:   File stem for saving (no extension). Ignored when ``ax`` is provided.
        ax:          Existing Axes for embedding inside a grid. Skips figure creation,
                     ``tight_layout``, and save — the caller manages those.
    """
    col_key, col_label = _parse_quality_col(quality_col)

    _own_fig = ax is None
    if _own_fig:
        fig, ax = plt.subplots(figsize=(9, 6))
    else:
        fig = ax.figure

    # Discover (arch, backend) combinations present in stats_df, optionally filtered
    unique_combos = sorted(
        {
            (arch, backend)
            for _, backend, arch in stats_df.index
            if (archs is None or arch in archs)
        }
    )

    for arch, backend in unique_combos:
        bpp_col = bpp_fpga_col if backend == "fpga" else bpp_gpu_col
        try:
            slice_ = stats_df.xs((backend, arch), level=("backend", "arch"))
        except KeyError:
            continue
        bpp_note = "bitstream" if "bitstream" in bpp_col else "likelihood"
        plot_rd_curve(
            ax,
            slice_,
            backend,
            arch,
            bpp_col,
            col_key,
            label=f"{arch} {backend.upper()} ({bpp_note})",
            annotate_lambda=annotate_lambda,
        )

    bpp_note = (
        "bpp = likelihood (GPU) / rANS (FPGA)"
        if "likelihood" in bpp_gpu_col
        else "bpp = rANS bitstream"
    )
    ax.set_xlabel(f"Bit-rate [bpp]  —  {bpp_note}", fontsize=8 if not _own_fig else 10)
    ax.set_ylabel(col_label, fontsize=8 if not _own_fig else 10)
    if title or _own_fig:
        ax.set_title(title or col_label, fontsize=8.5 if not _own_fig else 11)
    ax.legend(loc="lower right", fontsize=7 if not _own_fig else 9)
    ax.grid(alpha=0.3)

    if _own_fig:
        plt.tight_layout()
        if save_name and SAVE_FIGURES:
            out = PLOTS_DIR / f"{save_name}.pdf"
            fig.savefig(out, bbox_inches="tight")
            print(f"Saved: {out}")
    return fig

In [ ]:
# ── Config ────────────────────────────────────────────────────────────────────
ARCHS_TO_PLOT = ["ResSHyp", "SHyp", "ResFP", "FP"]  # reduce list for less crowded plots
# ─────────────────────────────────────────────────────────────────────────────
make_rd_figure(
    stats_df,
    archs=ARCHS_TO_PLOT,
    quality_col=("psnr_merlin", "PSNR vs MERLIN [dB]"),
    bpp_gpu_col=BPP_GPU_COL,
    annotate_lambda=False,
    save_name="RD-curves_gpu_vs_fpga_psnr",
)
plt.show()

In [ ]:
# ── Config ────────────────────────────────────────────────────────────────────
ARCHS_TO_PLOT = ["ResSHyp", "SHyp", "ResFP", "FP"]  # reduce list for less crowded plots
# ─────────────────────────────────────────────────────────────────────────────
make_rd_figure(
    stats_df,
    archs=ARCHS_TO_PLOT,
    quality_col=("ssim_merlin", "SSIM vs MERLIN"),
    bpp_gpu_col=BPP_GPU_COL,
    annotate_lambda=False,
    save_name="RD-curves_gpu_vs_fpga_ssim",
)
plt.show()

## 7 · FPGA degradation Δ plot

Key quantization question: **how much quality does the FPGA lose vs the GPU model?**

A per-seed FPGA − GPU delta is computed for each λ, then aggregated (mean ± min/max).
Negative values → FPGA is worse than GPU.

Set `ARCHS_FOR_DELTA` in the call cell below to control which architectures are shown.
Set `DELTA_METRICS` to control which quality metrics appear as grouped bars.

In [ ]:
def compute_delta(df: pd.DataFrame, quality_col: str = "psnr_merlin") -> pd.DataFrame:
    """Compute per-seed FPGA − GPU delta for a quality metric, then aggregate.

    Returns a DataFrame indexed by lambda with columns mean/std/min/max.
    """
    pivot = df.pivot_table(index=["lambda", "seed"], columns="backend", values=quality_col)
    pivot = pivot.dropna(subset=["gpu", "fpga"])
    pivot["delta"] = pivot["fpga"] - pivot["gpu"]
    return pivot["delta"].groupby("lambda").agg(["mean", "std", "min", "max"]).sort_index()


def plot_delta(
    deltas: List[Tuple[pd.DataFrame, str]],
    bpp_means: pd.Series,
    arch: str,
    yaxis_label: Optional[str] = None,
    save_name: Optional[str] = None,
) -> plt.Figure:
    """Bar chart of mean FPGA degradation per λ.

    Args:
        deltas:     List of ``(delta_df, label)`` tuples from ``compute_delta()``.
                    Single element → per-bar sign colouring.
                    Multiple elements → grouped bars, one colour per metric.
        bpp_means:  Mean GPU BPP per λ (for X-tick labels).
        save_name:  File stem for saving (no extension).
    """
    n_metrics = len(deltas)
    all_lambdas = sorted(set().union(*[set(d.index) for d, _ in deltas]))
    lambdas = np.array(all_lambdas, dtype=float)
    x = np.arange(len(lambdas))
    group_w = 0.70
    width = group_w / n_metrics
    offsets = np.linspace(-group_w / 2 + width / 2, group_w / 2 - width / 2, n_metrics)

    _metric_palette = [METRIC_COLORS["psnr"], METRIC_COLORS["ssim"], METRIC_COLORS["epd"]]

    fig, ax = plt.subplots(figsize=(max(10, 1.5 * len(lambdas)), 4))

    for i, (delta, label) in enumerate(deltas):
        means = np.array(
            [delta.loc[l, "mean"] if l in delta.index else float("nan") for l in lambdas]
        )
        mins = np.array(
            [delta.loc[l, "min"] if l in delta.index else float("nan") for l in lambdas]
        )
        maxs = np.array(
            [delta.loc[l, "max"] if l in delta.index else float("nan") for l in lambdas]
        )
        pos = x + offsets[i]

        bar_color = (
            BACKEND_COLORS["fpga"] if n_metrics == 1 else _metric_palette[i % len(_metric_palette)]
        )
        ax.bar(pos, means, width, color=bar_color, alpha=0.75, label=label)
        ax.errorbar(
            pos,
            means,
            yerr=[np.nan_to_num(means - mins), np.nan_to_num(maxs - means)],
            fmt="none",
            color=METRIC_COLORS["error_bars"],
            capsize=3,
            alpha=0.7,
        )

    ax.axhline(0, color="black", linewidth=0.8, linestyle="--")
    ax.set_xticks(x)
    ax.set_xticklabels(
        [f"λ={int(l)}\n({bpp_means.get(l, float('nan')):.3f} bpp)" for l in lambdas],
        fontsize=9,
    )
    ax.set_ylabel(yaxis_label or "Difference FPGA − GPU")
    ax.set_title(
        f"FPGA quantization degradation  —  {arch}\n(negative = FPGA worse than GPU)",
        fontsize=10,
    )
    ax.grid(axis="y", alpha=0.3)
    if n_metrics > 1:
        ax.legend(fontsize=8)
    plt.tight_layout()
    if save_name and SAVE_FIGURES:
        out = PLOTS_DIR / f"{save_name}.pdf"
        fig.savefig(out, bbox_inches="tight")
        print(f"Saved: {out}")
    return fig


# ── Config ────────────────────────────────────────────────────────────────────
ARCHS_FOR_DELTA = ["ResSHyp", "SHyp", "ResFP", "FP"]
DELTA_METRICS = [
    ("psnr_merlin", "PSNR [dB]"),
    ("ssim_merlin", "SSIM"),
    ("epd_merlin", "EPD"),
]
# ─────────────────────────────────────────────────────────────────────────────
for _arch in ARCHS_FOR_DELTA:
    _arch_df = df.query(f"arch == '{_arch}'")
    if _arch_df.empty:
        print(f"{y}No data for {_arch} — skipping.{e}")
        continue

    _deltas = [(compute_delta(_arch_df, col), label) for col, label in DELTA_METRICS]
    _bpp_by_lam = _arch_df.query("backend == 'gpu'").groupby("lambda")[BPP_GPU_COL].mean()

    print(f"\n{_arch} — average FPGA degradation across all λ:")
    for (col, label), (delta, _) in zip(DELTA_METRICS, _deltas):
        print(f"  {label}: {delta['mean'].mean():.3f}")

    plot_delta(
        _deltas,
        _bpp_by_lam,
        arch=_arch,
        yaxis_label=f"Quality difference FPGA − GPU  —  {_arch}",
        save_name=f"FPGA_degradation_{_arch}",
    )
    plt.show()

## 8 · BPP comparison — likelihood vs rANS bitstream

The GPU trains with a soft entropy estimate (likelihood BPP from the entropy bottleneck).
The FPGA uses real rANS coding. This plot shows how they compare at each λ.

- **If `update_wandb_runs.py` has run and added `bpp_bitstream` to W&B**: both backends
  show hard BPP and the comparison is apples-to-apples.
- **Currently**: GPU shows likelihood BPP (soft, usually slightly lower than real coding)
  and FPGA shows rANS BPP. The gap is the coding overhead of the real entropy coder.

> To add `bpp_bitstream` to the GPU runs: complete `update_wandb_runs.py` and re-run cell 2.
> `BPP_GPU_COL` will automatically switch to `"bpp_bitstream"`.


In [ ]:
def plot_bpp_comparison(
    df: pd.DataFrame,
    arch: str,
    save_name: Optional[str] = None,
) -> plt.Figure:
    """Grouped bar chart: GPU likelihood BPP vs GPU bitstream BPP vs FPGA rANS BPP per λ."""
    bpp_stats = df.groupby(["lambda", "backend"]).agg(
        bpp_gpu_lik_mean=("bpp_likelihood", "mean"),
        bpp_gpu_lik_std=("bpp_likelihood", "std"),
        bpp_bitstream_mean=("bpp_bitstream", "mean"),
        bpp_bitstream_std=("bpp_bitstream", "std"),
    )

    lambdas = sorted(df["lambda"].unique())
    x = np.arange(len(lambdas))
    width = 0.25
    fig, ax = plt.subplots(figsize=(12, 5))

    gpu_lik_means, gpu_lik_stds = [], []
    gpu_bit_means, gpu_bit_stds = [], []
    fpga_means, fpga_stds = [], []

    for lam in lambdas:
        try:
            g_row = bpp_stats.loc[(lam, "gpu")]
            gpu_lik_means.append(g_row.get("bpp_gpu_lik_mean", float("nan")))
            gpu_lik_stds.append(g_row.get("bpp_gpu_lik_std", 0) or 0)
            gpu_bit_means.append(g_row.get("bpp_bitstream_mean", float("nan")))
            gpu_bit_stds.append(g_row.get("bpp_bitstream_std", 0) or 0)
        except KeyError:
            gpu_lik_means.append(float("nan"))
            gpu_lik_stds.append(0)
            gpu_bit_means.append(float("nan"))
            gpu_bit_stds.append(0)
        try:
            f_row = bpp_stats.loc[(lam, "fpga")]
            fpga_means.append(f_row.get("bpp_bitstream_mean", float("nan")))
            fpga_stds.append(f_row.get("bpp_bitstream_std", 0) or 0)
        except KeyError:
            fpga_means.append(float("nan"))
            fpga_stds.append(0)

    ax.bar(
        x - width,
        gpu_lik_means,
        width,
        label="GPU likelihood BPP",
        color=C["platforms"]["gpu_idle"],
        yerr=gpu_lik_stds,
        capsize=2,
    )
    ax.bar(
        x,
        gpu_bit_means,
        width,
        label="GPU bitstream BPP",
        color=C["platforms"]["gpu_dynamic"],
        yerr=gpu_bit_stds,
        capsize=2,
    )
    ax.bar(
        x + width,
        fpga_means,
        width,
        label="FPGA rANS BPP",
        color=C["platforms"]["fpga_dynamic"],
        yerr=fpga_stds,
        capsize=2,
    )

    ax.set_xticks(x)
    ax.set_xticklabels([f"λ={int(l)}" for l in lambdas], fontsize=8)
    ax.set_ylabel("Bit-rate [bpp]")
    ax.set_title(f"BPP comparison: GPU likelihood / GPU bitstream / FPGA rANS  —  {arch}")
    ax.legend()
    ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    if save_name and SAVE_FIGURES:
        out = PLOTS_DIR / f"{save_name}.pdf"
        fig.savefig(out, bbox_inches="tight")
        print(f"Saved: {out}")
    return fig


# ── Config ────────────────────────────────────────────────────────────────────
ARCHS_FOR_BPP = ["ResSHyp", "SHyp", "ResFP", "FP"]
# ─────────────────────────────────────────────────────────────────────────────
for _arch in ARCHS_FOR_BPP:
    _arch_df = df.query(f"arch == '{_arch}'")
    if _arch_df.empty:
        print(f"{y}No data for {_arch} — skipping.{e}")
        continue
    plot_bpp_comparison(_arch_df, arch=_arch, save_name=f"BPP_comparison_{_arch}")
    plt.show()

## 9 · Hamburg Tile Visualization

Visual comparison on the **1024 × 1024 Hamburg tile** across all selected λ values.

**Layout** — one reference row + one FPGA/GPU row-pair per architecture in `ARCHS_FOR_VIS`:
- 1 arch → 3 rows (refs / FPGA / GPU)
- 2 archs → 5 rows (refs / FPGA arch₁ / GPU arch₁ / FPGA arch₂ / GPU arch₂)

All images are log-I, clipped per-image to mean ± 3σ.
PSNR is recomputed on-the-fly from the `.npy` arrays vs the MERLIN reference.

In [ ]:
# ── Config ────────────────────────────────────────────────────────────────────
TILE_NAME = "Hamburg_[11000:12024-8500:9524]"
ARCHS_FOR_VIS = ["ResSHyp", "FP"]  # ["ResSHyp", "SHyp"] → 5 rows; ["ResSHyp"] → 3 rows
SHOW_LAMBDAS = [1, 5, 100, 1000]  # None → all 10 lambdas
SEED_FOR_VIS = 0
SHOW_ADAM_NOC = False
SHOW_MERLIN_DDS = False
SHOW_CLOSEUP_ROI = True
CROP_R0, CROP_R1 = 0, 200
CROP_C0, CROP_C1 = 100, 300
# ─────────────────────────────────────────────────────────────────────────────

REFS_DIR = ROOT_DIR / "data" / "visualization" / TILE_NAME
SUBPLOT_IN = 2.6  # inches per subplot


# ── Helpers ───────────────────────────────────────────────────────────────────
def _linA_to_logI(linA: np.ndarray) -> np.ndarray:
    return np.log(np.square(linA) + EPS)


def _compute_psnr_tile(recon: np.ndarray, ref: np.ndarray) -> float:
    r = torch.from_numpy(recon.astype(np.float32))
    t = torch.from_numpy(ref.astype(np.float32))
    return _src_psnr(r, t)


def _show(
    ax: plt.Axes,
    logI: np.ndarray,
    title: str,
    subtitle: Optional[str] = None,
    border_color: Optional[str] = None,
) -> None:
    clipped = clip_logI(logI, mean_std_norm=True, clip_factor=3)
    ax.imshow(clipped, cmap="gray", aspect="equal", interpolation="none")
    ax.set_title(title, fontsize=8, fontweight="bold", pad=3)
    if subtitle:
        ax.text(
            0.5,
            -0.03,
            subtitle,
            transform=ax.transAxes,
            fontsize=6.5,
            ha="center",
            va="top",
            color="dimgray",
        )
    ax.axis("off")
    if border_color:
        for spine in ax.spines.values():
            spine.set_visible(True)
            spine.set_edgecolor(border_color)
            spine.set_linewidth(2.0)


def _crop(arr: np.ndarray) -> np.ndarray:
    return arr[CROP_R0:CROP_R1, CROP_C0:CROP_C1]


# ── Load reference images ─────────────────────────────────────────────────────
noisy_linA = np.load(REFS_DIR / "linA_Noisy.npy")
merlin_linA = np.load(REFS_DIR / "linA_MERLIN.npy")
adam_noc_linA = np.load(REFS_DIR / "linA_ADAM_NOC.npy") if SHOW_ADAM_NOC else None
merlin_dds_linA = np.load(REFS_DIR / "linA_MERLIN_DDS.npy") if SHOW_MERLIN_DDS else None

ref_panels: List[Tuple[str, np.ndarray]] = [("Noisy", noisy_linA), ("MERLIN", merlin_linA)]
if adam_noc_linA is not None:
    ref_panels.append(("ADAM-NOC", adam_noc_linA))
if merlin_dds_linA is not None:
    ref_panels.append(("MERLIN-DDS", merlin_dds_linA))


# ── Load FPGA & GPU reconstructions ──────────────────────────────────────────
# tile_data[arch][lmbda][backend] = {"linA", "logI", "psnr_merlin", "bpp"} or None
sel_lambdas: List[float] = sorted(SHOW_LAMBDAS) if SHOW_LAMBDAS else sorted(df["lambda"].unique())

tile_data: Dict[str, Dict[float, Dict[str, Optional[Dict]]]] = {}

for _arch in ARCHS_FOR_VIS:
    tile_data[_arch] = {}
    for lmbda in sel_lambdas:
        tile_data[_arch][lmbda] = {}
        for backend in ["fpga", "gpu"]:
            rows = df[
                (df["lambda"] == lmbda)
                & (df["seed"] == SEED_FOR_VIS)
                & (df["backend"] == backend)
                & (df["arch"] == _arch)
            ]
            if len(rows) == 0:
                tile_data[_arch][lmbda][backend] = None
                continue
            row = rows.iloc[0]

            if backend == "fpga":
                npy_path = Path(row["model_dir"]) / "results" / f"{TILE_NAME}_recon_linA.npy"
                metrics_path = Path(row["model_dir"]) / "results" / f"{TILE_NAME}_metrics.json"
            else:
                npy_path = Path(row["run_dir"]) / f"recon_{TILE_NAME}_linA.npy"
                metrics_path = Path(row["run_dir"]) / f"recon_{TILE_NAME}_metrics.json"

            if not npy_path.exists():
                print(
                    f"  {y}MISSING{e}: λ={int(lmbda)} seed={SEED_FOR_VIS} {backend} {_arch} — {npy_path.name}"
                )
                tile_data[_arch][lmbda][backend] = None
                continue

            linA = np.load(npy_path)
            psnr = _compute_psnr_tile(linA, merlin_linA)
            bpp = float("nan")
            if metrics_path.exists():
                m = json.loads(metrics_path.read_text())
                bpp = float(m.get("bpp" if backend == "fpga" else "bpp_bitstream", float("nan")))

            tile_data[_arch][lmbda][backend] = {
                "linA": linA,
                "logI": _linA_to_logI(linA),
                "psnr_merlin": psnr,
                "bpp": bpp,
            }

n_present = sum(
    1
    for arch_d in tile_data.values()
    for lam_d in arch_d.values()
    for v in lam_d.values()
    if v is not None
)
print(
    f"Loaded {n_present} / {2 * len(ARCHS_FOR_VIS) * len(sel_lambdas)} tiles  "
    f"(seed={SEED_FOR_VIS}, {len(ARCHS_FOR_VIS)} arch × {len(sel_lambdas)} λ × 2 backends)"
)

In [ ]:
n_refs = len(ref_panels)
n_lam = len(sel_lambdas)
n_cols = max(n_refs, n_lam)
n_rows = 1 + 2 * len(ARCHS_FOR_VIS)

# Row headers: one pair (FPGA, GPU) per architecture
_ROW_HEADERS = [(0, "References", "black")]
for _i, _arch in enumerate(ARCHS_FOR_VIS):
    _suffix = f"\n{_arch}" if len(ARCHS_FOR_VIS) > 1 else ""
    _ROW_HEADERS += [
        (1 + 2 * _i, f"FPGA{_suffix}\n(seed={SEED_FOR_VIS})", BACKEND_COLORS["fpga"]),
        (2 + 2 * _i, f"GPU{_suffix}\n(seed={SEED_FOR_VIS})", BACKEND_COLORS["gpu"]),
    ]

fig, axes = plt.subplots(
    n_rows,
    n_cols,
    figsize=(SUBPLOT_IN * n_cols + 0.5, SUBPLOT_IN * n_rows + 0.7),
    squeeze=False,
)

for _ri, _label, _color in _ROW_HEADERS:
    axes[_ri, 0].text(
        -0.12,
        0.5,
        _label,
        transform=axes[_ri, 0].transAxes,
        rotation=90,
        va="center",
        ha="right",
        fontsize=9,
        fontweight="bold",
        color=_color,
    )

# ── Row 0: References ─────────────────────────────────────────────────────────
for ci, (ref_name, ref_linA) in enumerate(ref_panels):
    _show(axes[0, ci], _linA_to_logI(ref_linA), ref_name)
    if SHOW_CLOSEUP_ROI and ref_name == "Noisy":
        axes[0, ci].add_patch(
            Rectangle(
                (CROP_C0, CROP_R0),
                CROP_C1 - CROP_C0,
                CROP_R1 - CROP_R0,
                linewidth=1.5,
                edgecolor="red",
                facecolor="none",
                linestyle="--",
            )
        )
for ci in range(n_refs, n_cols):
    axes[0, ci].axis("off")

# ── FPGA + GPU rows (one pair per architecture) ────────────────────────────────
for _i, _arch in enumerate(ARCHS_FOR_VIS):
    for _j, backend in enumerate(["fpga", "gpu"]):
        row_idx = 1 + 2 * _i + _j
        color = BACKEND_COLORS[backend]
        for ci, lmbda in enumerate(sel_lambdas):
            data = tile_data[_arch].get(lmbda, {}).get(backend)
            if data is None:
                axes[row_idx, ci].axis("off")
                axes[row_idx, ci].text(
                    0.5,
                    0.5,
                    "N/A",
                    ha="center",
                    va="center",
                    transform=axes[row_idx, ci].transAxes,
                    fontsize=10,
                    color="gray",
                )
                continue
            bpp_str = f"{data['bpp']:.3f}" if not np.isnan(data["bpp"]) else "N/A"
            subtitle = f"BPP={bpp_str}  PSNR={data['psnr_merlin']:.2f}dB"
            _show(axes[row_idx, ci], data["logI"], f"λ={int(lmbda)}", subtitle, border_color=color)
        for ci in range(n_lam, n_cols):
            axes[row_idx, ci].axis("off")

arch_str = " + ".join(ARCHS_FOR_VIS)
fig.suptitle(
    f"Hamburg Tile  —  {TILE_NAME}\n"
    f"log-I · per-image mean±3σ  |  PSNR vs MERLIN  |  seed={SEED_FOR_VIS}  |  {arch_str}",
    fontsize=10,
    y=1.01,
)
plt.tight_layout(h_pad=1.2, w_pad=0.3)

if SAVE_FIGURES:
    out = PLOTS_DIR / f"Visualizations_Hamburg_{'_'.join(ARCHS_FOR_VIS)}.pdf"
    fig.savefig(out, bbox_inches="tight")
    print(f"Saved: {out}")
plt.show()

## 9-bis · Hamburg Tile — Close-up

In [ ]:
n_refs_cu = len(ref_panels)
n_lam_cu = len(sel_lambdas)
n_cols_cu = max(n_refs_cu, n_lam_cu)
n_rows_cu = 1 + 2 * len(ARCHS_FOR_VIS)

# Reuse the same row headers and colouring as the full-tile figure
_ROW_HEADERS_CU = [(0, "References", "black")]
for _i, _arch in enumerate(ARCHS_FOR_VIS):
    _suffix = f"\n{_arch}" if len(ARCHS_FOR_VIS) > 1 else ""
    _ROW_HEADERS_CU += [
        (1 + 2 * _i, f"FPGA{_suffix}\n(seed={SEED_FOR_VIS})", BACKEND_COLORS["fpga"]),
        (2 + 2 * _i, f"GPU{_suffix}\n(seed={SEED_FOR_VIS})", BACKEND_COLORS["gpu"]),
    ]

fig_cu, axes_cu = plt.subplots(
    n_rows_cu,
    n_cols_cu,
    figsize=(SUBPLOT_IN * n_cols_cu + 0.5, SUBPLOT_IN * n_rows_cu + 0.7),
    squeeze=False,
)

for _ri, _label, _color in _ROW_HEADERS_CU:
    axes_cu[_ri, 0].text(
        -0.12,
        0.5,
        _label,
        transform=axes_cu[_ri, 0].transAxes,
        rotation=90,
        va="center",
        ha="right",
        fontsize=9,
        fontweight="bold",
        color=_color,
    )

# Row 0: References (cropped)
for ci, (ref_name, ref_linA) in enumerate(ref_panels):
    _show(axes_cu[0, ci], _crop(_linA_to_logI(ref_linA)), ref_name)
for ci in range(n_refs_cu, n_cols_cu):
    axes_cu[0, ci].axis("off")

# FPGA + GPU rows (cropped)
for _i, _arch in enumerate(ARCHS_FOR_VIS):
    for _j, backend in enumerate(["fpga", "gpu"]):
        row_idx = 1 + 2 * _i + _j
        color = BACKEND_COLORS[backend]
        for ci, lmbda in enumerate(sel_lambdas):
            data = tile_data[_arch].get(lmbda, {}).get(backend)
            if data is None:
                axes_cu[row_idx, ci].axis("off")
                axes_cu[row_idx, ci].text(
                    0.5,
                    0.5,
                    "N/A",
                    ha="center",
                    va="center",
                    transform=axes_cu[row_idx, ci].transAxes,
                    fontsize=10,
                    color="gray",
                )
                continue
            bpp_str = f"{data['bpp']:.3f}" if not np.isnan(data["bpp"]) else "N/A"
            subtitle = f"BPP={bpp_str}  PSNR={data['psnr_merlin']:.2f}dB"
            _show(
                axes_cu[row_idx, ci],
                _crop(data["logI"]),
                f"λ={int(lmbda)}",
                subtitle,
                border_color=color,
            )
        for ci in range(n_lam_cu, n_cols_cu):
            axes_cu[row_idx, ci].axis("off")

arch_str = " + ".join(ARCHS_FOR_VIS)
fig_cu.suptitle(
    f"Hamburg Tile — Close-up  rows [{CROP_R0}:{CROP_R1}]  cols [{CROP_C0}:{CROP_C1}]\n"
    f"log-I · per-image mean±3σ  |  PSNR vs MERLIN  |  seed={SEED_FOR_VIS}  |  {arch_str}",
    fontsize=10,
    y=1.01,
)
plt.tight_layout(h_pad=1.2, w_pad=0.3)

if SAVE_FIGURES:
    out_cu = PLOTS_DIR / f"Visualizations_Hamburg_closeup_{'_'.join(ARCHS_FOR_VIS)}.pdf"
    fig_cu.savefig(out_cu, bbox_inches="tight")
    print(f"Saved: {out_cu}")
plt.show()

## 10 · All-metrics RD-curve grid

Full-quality overview across **all metrics** in `_METRIC_LABELS` for a selected architecture.
Edit `ARCHS_TO_PLOT` and the `_METRIC_LABELS` dict in the cell below to control the output.

In [ ]:
# Human-readable Y-axis labels.  Comment out entries to hide them from the grid.
_METRIC_LABELS: Dict[str, str] = {
    "psnr_merlin": "PSNR vs MERLIN [dB]",
    "ssim_merlin": "SSIM vs MERLIN",
    "epd_merlin": "EPD vs MERLIN",
    "ratio_mean": "Mean(noisy_I / recon_I)",
    "ratio_enl": "ENL of Ratio image",
    "enl_recon": "ENL of Reconstruction",
    # "psnr_adam_noc": "PSNR vs ADAM-NOC [dB]",
    # "epd_adam_noc":  "EPD vs ADAM-NOC",
}
N_COLS_GRID = 3  # 2 or 3 — figure grows taller rather than wider

# ── Config ────────────────────────────────────────────────────────────────────
ARCHS_TO_PLOT = ["ResSHyp", "SHyp", "ResFP", "FP"]
# ─────────────────────────────────────────────────────────────────────────────
n_metrics = len(_METRIC_LABELS)
n_rows_grid = (n_metrics + N_COLS_GRID - 1) // N_COLS_GRID

for _arch in ARCHS_TO_PLOT:
    _arch_df = df.query(f"arch == '{_arch}'")
    if _arch_df.empty:
        print(f"{y}No data for {_arch} — skipping.{e}")
        continue

    fig_all, axes_all = plt.subplots(
        n_rows_grid,
        N_COLS_GRID,
        figsize=(N_COLS_GRID * 5.5, n_rows_grid * 4.2),
        squeeze=False,
    )

    for idx, (metric_col, metric_label) in enumerate(_METRIC_LABELS.items()):
        _r, _c = divmod(idx, N_COLS_GRID)
        make_rd_figure(
            stats_df,  # use aggregated stats (MultiIndex), not raw _arch_df
            quality_col=(metric_col, metric_label),
            bpp_gpu_col=BPP_GPU_COL,
            archs=[_arch],
            annotate_lambda=False,
            ax=axes_all[_r, _c],
        )

    for idx in range(n_metrics, n_rows_grid * N_COLS_GRID):
        _r, _c = divmod(idx, N_COLS_GRID)
        axes_all[_r, _c].axis("off")

    fig_all.suptitle(
        f"GPU vs FPGA — All Quality Metrics  —  {_arch}",
        fontsize=12,
        fontweight="bold",
    )
    plt.tight_layout()

    if SAVE_FIGURES:
        out_all = PLOTS_DIR / f"rd_all_metrics_{_arch}.pdf"
        fig_all.savefig(out_all, dpi=150, bbox_inches="tight")
        print(f"Saved: {out_all}")
    plt.show()